# Crypto cross-sectional return forecasting: a signal that is real, and a Sharpe that is not proven

**EN:** Every day at 00:00 UTC I rank the ~30 most liquid coins on Binance by how I expect them to do over the next 24 hours compared with each other. I then check whether that ranking is worth anything as a long-short portfolio, after costs and after correcting for how many things I tried.

**TR:** Her gün 00:00 UTC'de Binance'teki en likit ~30 coini, önümüzdeki 24 saatte birbirlerine göre nasıl performans göstereceklerini beklediğime göre sıralıyorum. Sonra bu sıralamanın, maliyetlerden sonra ve kaç şey denediğimi düzelttikten sonra, long-short bir portföy olarak bir değeri olup olmadığını kontrol ediyorum.

| | |
|---|---|
| Data / Veri | Binance spot, hourly, 2019-01 → 2026-08, 257 coins (65 of them delisted along the way) |
| Universe / Evren | Top 30 by 30-day USDT volume, re-picked every month, point in time |
| Target / Hedef | Log return from t+1h to t+25h (one hour of execution lag) |
| Validation / Doğrulama | Walk-forward, 10 folds, retrain every 6 months, 2-day embargo, test 2022-01 → 2026-08 |
| Models / Modeller | Baselines (random, momentum, reversal, Ridge) → LightGBM, GRU, Transformer, + quantile versions |

**What I found / Bulduklarım**
1. **EN:** The ranking signal is real: rank IC ≈ 0.11–0.12 (Newey-West t ≈ 17), a beta-hedged alpha of ≈ 40 bps/day (t ≈ 5.6), and an IC that is positive in every calendar year.
   **TR:** Sıralama sinyali gerçek: rank IC ≈ 0,11–0,12 (Newey-West t ≈ 17), beta'dan arındırılmış günlük ≈ 40 bps alpha (t ≈ 5,6) ve her takvim yılında pozitif bir IC.
2. **EN:** Most of that IC is one well-known effect: high-volatility "lottery" coins have a terrible *median* day. Rank IC rewards the median; a portfolio earns the *mean*. That is why IC 0.11 turns into a gross Sharpe of only ~2.3.
   **TR:** O IC'nin çoğu bilinen tek bir etki: yüksek volatiliteli "piyango" coinlerin *medyan* günü berbat. Rank IC medyanı ödüllendiriyor; portföy ise *ortalamayı* kazanıyor. IC 0,11'in yalnızca ~2,3 gross Sharpe'a dönüşmesinin sebebi bu.
3. **EN:** Non-linear models double the monetisable alpha of a linear model on the same features (40 vs 18 bps/day). The hourly sequence models (GRU, Transformer) do **not** beat LightGBM on a daily snapshot.
   **TR:** Doğrusal olmayan modeller, aynı özelliklerle doğrusal bir modelin paraya çevrilebilir alpha'sını iki katına çıkarıyor (günlük 40'a karşı 18 bps). Saatlik dizi modelleri (GRU, Transformer), günlük anlık görüntü üzerindeki LightGBM'i **geçemiyor**.
4. **EN:** Costs decide everything. Daily turnover of 1.5 at 15 bps/side eats half the gross Sharpe. A pre-declared 3-day smoothing cuts turnover by 70% and lifts net Sharpe from 1.1 to 1.45, but after deflating for the 12 strategies I evaluated, the **deflated Sharpe is ~0.12**: I cannot claim the net edge is not luck.
   **TR:** Her şeyi maliyetler belirliyor. Taraf başına 15 bps ile günlük 1,5 turnover, gross Sharpe'ın yarısını yiyor. Önceden sabitlenmiş 3 günlük düzleştirme turnover'ı %70 azaltıp net Sharpe'ı 1,1'den 1,45'e çıkarıyor; ama değerlendirdiğim 12 strateji için deflasyondan sonra **deflated Sharpe ~0,12**: net avantajın şans olmadığını iddia edemem.
5. **EN:** The quantile models are well calibrated (80% interval covers 80%) but beat a simple volatility-scaled baseline by only ~2% in pinball loss.
   **TR:** Quantile modelleri iyi kalibre (%80'lik aralık %80'i kapsıyor), ama basit bir volatiliteyle ölçeklenmiş baseline'ı pinball kaybında yalnızca ~%2 geçiyor.

## 0. Setup and device check / Kurulum ve cihaz kontrolü

**EN:** I print the device explicitly. My GPU is a GTX 1050 Ti (Pascal, sm_61), which needs the cu126 build of PyTorch; the arch list below proves the build and the card match.

**TR:** Cihazı açıkça yazdırıyorum. GPU'm bir GTX 1050 Ti (Pascal, sm_61) ve PyTorch'un cu126 derlemesini istiyor; aşağıdaki mimari listesi derlemeyle kartın eşleştiğini kanıtlıyor.

In [1]:
import sys, warnings, subprocess, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src import config, backtest as B, metrics as M, walkforward as W, evaluate as E
from src.data import load_panel
from src.features import build_samples, COIN_FEATURES, ALL_FEATURES
from src.splits import walk_forward_folds
from src.train import get_device, set_seed
from scripts.run_walkforward import load_universe, load_result

pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
set_seed(config.SEED)
device = get_device()

torch             : 2.7.1+cu126
device            : cuda
gpu               : NVIDIA GeForce GTX 1050 Ti
compute capability: sm_61
vram              : 4.0 GB
built for archs   : ['sm_50', 'sm_60', 'sm_61', 'sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90']


## 1. Data and the point-in-time universe / Veri ve point-in-time evren

**EN:** `scripts/download_data.py` pulls Binance's public archive (data.binance.vision), not the REST API, because the archive keeps delisted pairs. It first downloads daily bars for all ~590 candidate USDT pairs, picks the top 30 by volume each month using only the 30 days before that month, and then downloads hourly bars only for coins that were ever picked. Below: how much the universe churns.

**TR:** `scripts/download_data.py`, REST API'yi değil Binance'in herkese açık arşivini (data.binance.vision) kullanıyor; çünkü arşiv delist edilmiş pariteleri saklıyor. Önce ~590 aday USDT paritesinin günlük barlarını indiriyor, her ay yalnızca o aydan önceki 30 güne bakarak hacme göre ilk 30'u seçiyor ve sonra saatlik barları yalnızca bir kez olsun seçilmiş coinler için indiriyor. Aşağıda evrenin ne kadar değiştiğini görüyoruz.

In [2]:
hourly = load_panel("hourly")
universe = load_universe()
utable = pd.read_csv(config.REPORTS_DIR / "universe.csv", parse_dates=["month"])
quality = pd.read_csv(config.REPORTS_DIR / "data_quality.csv", parse_dates=["first", "last"], index_col="symbol")

sizes = utable.groupby("month").size()
end = pd.Timestamp(config.DATA_END_MONTH) + pd.offsets.MonthEnd(0)
delisted = quality[quality["last"] < end - pd.Timedelta(days=7)]
print(f"hourly panel      : {len(hourly.index):,} hours x {len(hourly.symbols)} symbols")
print(f"universe months   : {len(sizes)} | coins per month min {sizes.min()}, median {sizes.median():.0f}")
print(f"distinct coins    : {utable['symbol'].nunique()} ever in the top {config.UNIVERSE_SIZE}")
print(f"stopped trading   : {len(delisted)} coins, e.g. {', '.join(s.replace('USDT','') for s in delisted.index[:15])}")
print("LUNA in the universe:", [m.strftime('%Y-%m') for m in utable.loc[utable.symbol == 'LUNAUSDT', 'month']][:16])
E.plot_universe(utable);

hourly panel      : 67,200 hours x 257 symbols
universe months   : 89 | coins per month min 22, median 30
distinct coins    : 257 ever in the top 30
stopped trading   : 65 coins, e.g. EOS, BCHSV, BCHABC, NULS, WAVES, BTT, XMR, NANO, OMG, MITH, MATIC, FTM, ERD, NPXS, TOMO
LUNA in the universe: ['2021-03', '2021-04', '2021-06', '2021-07', '2021-08', '2021-09', '2021-10', '2021-11', '2021-12', '2022-01', '2022-02', '2022-03', '2022-04', '2022-05', '2022-10', '2022-11']


**EN:** LUNA is in the universe until May 2022, when it collapsed (it comes back from October 2022 as LUNA 2.0, a different token under the same ticker: see defect 2 below). A backtest that picked "the top 30 coins" with today's list would never have held it. That is survivorship bias, and this universe avoids it by construction.

**TR:** LUNA, çöktüğü Mayıs 2022'ye kadar evrende (Ekim 2022'den itibaren aynı sembolle farklı bir token olan LUNA 2.0 olarak geri geliyor: aşağıdaki 2. kusura bakın). "İlk 30 coin"i bugünün listesiyle seçen bir backtest onu asla tutmazdı. Buna survivorship bias deniyor ve bu evren onu tasarım gereği önlüyor.

### Three data defects I found and fixed / Bulup düzelttiğim üç veri kusuru

1. **EN:** *A stablecoin with an unknown name.* `U` (2026) is pegged at 1.00 but was not on my stablecoin list and entered the top 30. I added a data-driven guard: a coin whose daily volatility over the ranking window is below 0.5% cannot enter, whatever its name.
   **TR:** *Adı bilinmeyen bir stablecoin.* `U` (2026) 1,00'a sabitli ama stablecoin listemde değildi ve ilk 30'a girdi. Veriye dayalı bir koruma ekledim: sıralama penceresindeki günlük volatilitesi %0,5'in altında olan bir coin, adı ne olursa olsun evrene giremiyor.
2. **EN:** *Token swaps reusing a ticker.* After multi-day trading halts, six tickers came back as a different asset: LUNA relaunched as LUNA 2.0 (+12 log return in one "hour"), SUN, BNX, STRAX and BTCST were redenominated. A 30-day momentum feature spanning the gap would read a swap as a 100,000x move. Rule: a halt of ≥ 72 hours breaks the series; no sample is built until a full 30-day warm-up has passed, and the universe requires 60 days of *continuous* trading.
   **TR:** *Sembolü yeniden kullanan token swap'ları.* Çok günlü işlem durdurmalarından sonra altı sembol farklı bir varlık olarak geri geldi: LUNA, LUNA 2.0 olarak yeniden çıktı (tek bir "saatte" +12 log getiri); SUN, BNX, STRAX ve BTCST redenominasyona uğradı. Boşluğu aşan 30 günlük bir momentum özelliği, bir swap'ı 100.000 kat hareket olarak okurdu. Kural: ≥ 72 saatlik bir durdurma seriyi kesiyor; tam 30 günlük ısınma geçene kadar örnek kurulmuyor ve evren 60 günlük *kesintisiz* işlem istiyor.
3. **EN:** *Missing bars and timestamp units.* Binance switched spot files from milliseconds to microseconds in 2025, which silently lands in the year 50,000 if read naively. Missing hours are never interpolated: an invented price is an invented return. Any window that touches a missing hour is dropped.
   **TR:** *Eksik barlar ve zaman damgası birimleri.* Binance 2025'te spot dosyalarını milisaniyeden mikrosaniyeye geçirdi; saf bir okumayla bu, sessizce 50.000'li yıllara düşüyor. Eksik saatler asla interpole edilmiyor: uydurulmuş bir fiyat, uydurulmuş bir getiridir. Eksik bir saate dokunan her pencere atılıyor.

In [3]:
# EN: the token-swap evidence, recomputed from the raw panel.
# TR: token swap kanıtı, ham panelden yeniden hesaplanmış hâli.
logp = np.log(hourly.close.ffill())
jumps = logp.diff().where(hourly.traded)
halted = hourly.traded.rolling(config.LONG_GAP_HOURS).sum().eq(0).shift(1, fill_value=False)
rows = []
for sym in hourly.symbols:
    after_gap = hourly.traded[sym] & halted[sym]
    for t in hourly.index[after_gap.to_numpy()]:
        rows.append({"symbol": sym, "resumed": t, "log_jump": jumps.at[t, sym]})
swaps = pd.DataFrame(rows)
swaps[swaps["log_jump"].abs() > 1].sort_values("resumed")

,symbol,resumed,log_jump
49,BTCSTUSDT,2021-03-19 08:00:00,-2.089
202,SUNUSDT,2021-06-18 05:00:00,-6.838
129,LUNAUSDT,2022-05-31 07:00:00,12.050
44,BNXUSDT,2023-02-22 09:00:00,-4.755
197,STRAXUSDT,2024-03-28 09:00:00,-2.227


## 2. Features, target and the leakage rules / Özellikler, hedef ve sızıntı kuralları

**EN:** One row per (day, coin). The rule is absolute: features use bars that closed at or before 00:00; the target is the log return from 01:00 to 01:00 the next day. The one-hour gap is deliberate: a signal computed at 00:00 cannot also be filled at 00:00.
- 12 coin features: returns over 1h to 30d, realised volatility, high-low range, volume shock, Amihud illiquidity, 30-day beta to BTC.
- 3 market features (identical for every coin on a day): BTC 1d/7d return and volatility.
- A cross-sectional rank of every coin feature, so the model sees "how volatile compared with today's peers", which is stable across regimes.
- For GRU/Transformer additionally: the last 120 hourly steps of 5 raw channels.

**TR:** (gün, coin) başına bir satır. Kural kesin: özellikler 00:00'da ya da öncesinde kapanmış barları kullanıyor; hedef, 01:00'den ertesi gün 01:00'e kadarki log getiri. Bir saatlik boşluk bilinçli: 00:00'da hesaplanan bir sinyal 00:00'da gerçekleştirilemez.
- 12 coin özelliği: 1 saatten 30 güne kadar getiriler, gerçekleşmiş volatilite, high-low aralığı, hacim şoku, Amihud illikiditesi, BTC'ye göre 30 günlük beta.
- 3 piyasa özelliği (bir günde her coin için aynı): BTC'nin 1g/7g getirisi ve volatilitesi.
- Her coin özelliğinin kesitsel sırası; böylece model "bugünkü akranlarına göre ne kadar volatil" bilgisini görüyor ve bu, rejimler arasında kararlı.
- GRU/Transformer için ek olarak: 5 ham kanalın son 120 saatlik adımı.

In [4]:
t0 = time.time()
samples = build_samples(hourly, universe, start=config.FIRST_SAMPLE_DATE)
frame = samples.frame
per_day = frame.groupby(level="time").size()
print(f"{len(frame):,} rows over {len(per_day):,} days | coins per day median {per_day.median():.0f}, "
      f"min {per_day.min()} | built in {time.time() - t0:.0f}s")
frame[COIN_FEATURES[:6] + ["rank_vol_168h", "y_day0", "y_rankgauss"]].tail(4)

77,765 rows over 2,610 days | coins per day median 30, min 21 | built in 21s


ret_1h  ret_4h  ret_24h  ret_72h  ret_168h  ret_720h  rank_vol_168h  y_day0  y_rankgauss
time       symbol                                                                                           
2026-08-30 WLDUSDT   0.012   0.016    0.004   -0.085    -0.016     0.218          0.300  -0.047       -0.784
           XLMUSDT   0.003   0.003    0.002   -0.030    -0.084     0.045         -0.167  -0.023       -0.042
           XRPUSDT   0.002  -0.001    0.009   -0.019    -0.046     0.254         -0.033  -0.022        0.126
           ZECUSDT   0.002   0.010    0.050    0.031     0.048     0.582          0.267  -0.002        1.383

**EN:** I do not ask anyone to trust the rules above; the test suite checks them mechanically on a synthetic market where I planted the traps. The strongest test deletes all data after time t, recomputes every feature and demands identical values at t. I also mutation-tested the suite: planting a centred rolling window or dropping the missing-bar rule makes the relevant test fail.

**TR:** Kimseden yukarıdaki kurallara güvenmesini istemiyorum; test paketi onları, tuzakları benim yerleştirdiğim sentetik bir piyasa üzerinde mekanik olarak kontrol ediyor. En güçlü test, t'den sonraki tüm veriyi silip her özelliği yeniden hesaplıyor ve t anında birebir aynı değerleri istiyor. Test paketini mutasyonla da sınadım: ortalanmış bir kayan pencere yerleştirmek ya da eksik bar kuralını kaldırmak ilgili testi kırıyor.

In [5]:
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", str(PROJECT_ROOT / "tests")],
                     capture_output=True, text=True, cwd=PROJECT_ROOT)
print(out.stdout.strip().splitlines()[-1])

25 passed in 22.45s


## 3. Walk-forward folds / Walk-forward fold'ları

**EN:** Ten folds, an expanding training window, three months of validation right before each test half-year, and a two-day embargo between every pair of slices so that no 25-hour label straddles a boundary.

**TR:** On fold, genişleyen bir eğitim penceresi, her test yarıyılından hemen önce üç aylık doğrulama ve 25 saatlik hiçbir etiketin sınırı aşmaması için her iki dilim arasında iki günlük embargo.

In [6]:
folds = walk_forward_folds(config.FIRST_SAMPLE_DATE, config.FIRST_TEST_DATE,
                           frame.index.get_level_values("time").max())
for f in folds:
    print(f)

fold  1: train 2019-03-01..2021-09-27 | val 2021-09-30..2021-12-29 | test 2022-01-01..2022-06-30
fold  2: train 2019-03-01..2022-03-26 | val 2022-03-29..2022-06-28 | test 2022-07-01..2022-12-31
fold  3: train 2019-03-01..2022-09-27 | val 2022-09-30..2022-12-29 | test 2023-01-01..2023-06-30
fold  4: train 2019-03-01..2023-03-26 | val 2023-03-29..2023-06-28 | test 2023-07-01..2023-12-31
fold  5: train 2019-03-01..2023-09-27 | val 2023-09-30..2023-12-29 | test 2024-01-01..2024-06-30
fold  6: train 2019-03-01..2024-03-26 | val 2024-03-29..2024-06-28 | test 2024-07-01..2024-12-31
fold  7: train 2019-03-01..2024-09-27 | val 2024-09-30..2024-12-29 | test 2025-01-01..2025-06-30
fold  8: train 2019-03-01..2025-03-26 | val 2025-03-29..2025-06-28 | test 2025-07-01..2025-12-31
fold  9: train 2019-03-01..2025-09-27 | val 2025-09-30..2025-12-29 | test 2026-01-01..2026-06-30
fold 10: train 2019-03-01..2026-03-26 | val 2026-03-29..2026-06-28 | test 2026-07-01..2026-08-30


## 4. Baselines first / Önce baseline'lar

**EN:** Before any model is trained, the bar goes on screen. `Random` is the null: after costs it shows what trading alone does to a strategy. `Momentum7d` and `Reversal24h` are the textbook factors, and `Ridge` is a linear model on exactly the features the big models see.

**TR:** Herhangi bir model eğitilmeden önce çıta ekrana geliyor. `Random` sıfır hipotezi: maliyetlerden sonra tek başına işlem yapmanın bir stratejiye ne yaptığını gösteriyor. `Momentum7d` ve `Reversal24h` ders kitabı faktörleri, `Ridge` ise büyük modellerin gördüğü özelliklerin tam aynısıyla eğitilen doğrusal bir model.

In [7]:
base = W.run_walk_forward(samples, folds, W.POINT_BASELINES, device, verbose=False)
base_table, _ = W.score_point_models(base)
cols = ["ic_mean", "ic_tstat_nw", "sharpe_gross", "sharpe_net", "turnover"]
base_table[cols]

,ic_mean,ic_tstat_nw,sharpe_gross,sharpe_net,turnover
model,,,,,
Ridge,0.111,15.448,1.102,-0.058,1.490
Reversal24h,0.024,4.037,-0.026,-2.853,3.068
Random,-0.006,-1.348,-0.055,-4.227,3.224
Momentum7d,-0.015,-2.355,0.597,-0.436,1.233


**EN:** Ridge already reaches a rank IC of 0.11 with a Newey-West t-stat of 15, yet its long-short book barely breaks even before costs and loses after them. An IC that high with a PnL that weak is the first thing I had to explain before trusting anything else.

**TR:** Ridge şimdiden 0,11 rank IC'ye ve 15'lik Newey-West t-istatistiğine ulaşıyor; ama long-short portföyü maliyet öncesi zar zor başa baş, maliyet sonrası ise zarar ediyor. Bu kadar yüksek bir IC ile bu kadar zayıf bir PnL, başka herhangi bir şeye güvenmeden önce açıklamam gereken ilk şeydi.

## 5. Why IC is not PnL / IC neden PnL değil

**EN:** Single-feature ICs show where the signal lives: almost entirely in volatility. Then I split coins into volatility quintiles each day and compare the *mean* and the *median* next-day return (both relative to the day's cross-sectional mean).

**TR:** Tek özellikli IC'ler sinyalin nerede yaşadığını gösteriyor: neredeyse tamamen volatilitede. Sonra coinleri her gün volatilite quintile'larına ayırıp ertesi günün *ortalama* ve *medyan* getirisini karşılaştırıyorum (ikisi de günün kesitsel ortalamasına göre).

In [8]:
train_part = frame.loc[: pd.Timestamp(config.FIRST_TEST_DATE)]  # pre-test data only / yalnızca test öncesi veri
single = pd.Series({c: M.daily_rank_ic(train_part[c], train_part["y_day0"]).mean() for c in COIN_FEATURES})
print("single-feature rank IC (2019-2021):")
print(single.sort_values().round(3).to_string())

simple = np.expm1(train_part["y_day0"])
rel = simple - simple.groupby(level="time").transform("mean")
bucket = np.ceil(train_part.groupby(level="time")["vol_168h"].rank(pct=True) * 5).clip(1, 5)
ladder = pd.DataFrame({"mean_bps": rel.groupby(bucket).mean() * 1e4,
                       "median_bps": rel.groupby(bucket).median() * 1e4,
                       "p95_bps": rel.groupby(bucket).quantile(0.95) * 1e4})
ladder.index = [f"vol Q{int(i)}" for i in ladder.index]
ladder.round(1)

single-feature rank IC (2019-2021):
hl_range_24h   -0.096
vol_168h       -0.093
vol_24h        -0.090
amihud         -0.057
ret_4h         -0.038
ret_24h        -0.038
ret_1h         -0.026
beta_720h      -0.025
ret_72h        -0.025
volume_z       -0.011
ret_168h       -0.009
ret_720h       -0.006


,mean_bps,median_bps,p95_bps
vol Q1,1.4,-27.0,493.7
vol Q2,-4.4,-40.6,541.4
vol Q3,-0.6,-53.8,669.4
vol Q4,0.1,-70.1,827.0
vol Q5,3.5,-97.9,1166.0


**EN:** From the calmest to the wildest fifth of coins the *median* next-day return falls steeply (about -27 to -98 bps), while the *mean* barely moves, because the most volatile coins also produce the biggest pumps (look at the 95th percentile). Spearman IC scores the ordering of the typical day, so it sees the steep median slope. A long-short portfolio earns the average, pumps included, so it only sees the gentle mean slope. This is the crypto version of the "lottery stock" effect; it is also why I report the dollar-based metrics next to IC and never IC alone.

**TR:** En sakin beşte birden en vahşi beşte bire doğru ertesi günün *medyan* getirisi dik bir şekilde düşüyor (yaklaşık -27'den -98 bps'ye); *ortalama* ise neredeyse hiç kıpırdamıyor, çünkü en volatil coinler en büyük pump'ları da üretiyor (95. yüzdeliğe bakın). Spearman IC tipik günün sıralamasını puanlıyor, dolayısıyla dik medyan eğimini görüyor. Long-short bir portföy ise pump'lar dahil ortalamayı kazanıyor, dolayısıyla yalnızca hafif ortalama eğimini görüyor. Bu, "piyango hissesi" etkisinin kripto versiyonu; ayrıca IC'nin yanında dolar bazlı metrikleri raporlamamın ve IC'yi asla tek başına vermememin sebebi de bu.

## 6. Training one fold live / Tek bir fold'u canlı eğitmek

**EN:** The full walk-forward (10 folds x 11 models, including nine LightGBM quantile models per fold) takes ~70 minutes on my GTX 1050 Ti, so I run it with `scripts/run_walkforward.py` and load its predictions below. To prove those saved numbers come from this exact code, I retrain fold 1 here and compare. LightGBM is deterministic, so its predictions must match exactly; the GRU runs on the GPU (cuDNN is not bit-for-bit deterministic), so I compare its IC.

**TR:** Tam walk-forward (10 fold x 11 model, fold başına dokuz LightGBM quantile modeli dahil) GTX 1050 Ti'mde ~70 dakika sürüyor; bu yüzden onu `scripts/run_walkforward.py` ile koşturup tahminlerini aşağıda yüklüyorum. Kaydedilmiş bu sayıların tam olarak bu koddan geldiğini kanıtlamak için fold 1'i burada yeniden eğitip karşılaştırıyorum. LightGBM deterministik, dolayısıyla tahminleri birebir tutmalı; GRU ise GPU'da koşuyor (cuDNN bit bit deterministik değil), dolayısıyla onun IC'sini karşılaştırıyorum.

In [9]:
full = load_result("full")
live = W.run_walk_forward(samples, folds[:1], ["LightGBM", "GRU"], device, verbose=True)

saved = full.point.loc[live.point.index]
y = live.targets["y_day0"]
print("\nLightGBM max |live - saved| :", float((live.point["LightGBM"] - saved["LightGBM"]).abs().max()))
print(f"GRU fold-1 IC  live {M.daily_rank_ic(live.point['GRU'], y).mean():+.4f}  "
      f"saved {M.daily_rank_ic(saved['GRU'], y).mean():+.4f}")
E.plot_training_curves(live.histories, "GRU");


fold  1: train 2019-03-01..2021-09-27 | val 2021-09-30..2021-12-29 | test 2022-01-01..2022-06-30  (rows: train 24,282 / val 2,580 / test 5,398)


  [GRU] 19,585 parameters


    epoch  1/30  train 0.9527  val 0.9420  (6.4s) *


    epoch  2/30  train 0.9452  val 0.9388  (1.5s) *


    epoch  3/30  train 0.9447  val 0.9411  (1.5s)


...

    epoch  5/30  train 0.9392  val 0.9388  (1.5s)


    epoch  6/30  train 0.9372  val 0.9394  (1.5s)


    epoch  7/30  train 0.9363  val 0.9419  (1.5s)


  test IC: LightGBM +0.1597  GRU +0.1566

LightGBM max |live - saved| : 0.0


GRU fold-1 IC  live +0.1566  saved +0.1566


## 7. The out-of-sample scorecard / Örneklem dışı skor tablosu

**EN:** Test predictions of all ten folds are stitched into one history, January 2022 to August 2026, and every number below is computed on it. Columns: signal quality (IC, its Newey-West t-stat), the long-short book before and after 15 bps per side, turnover, the book's beta to the equal-weight universe and the alpha left after removing it, and two significance tests: PSR (probability the true Sharpe > 0, one trial) and the deflated Sharpe ratio (the same, corrected for the 12 candidate strategies I evaluated).

**TR:** On fold'un test tahminleri Ocak 2022'den Ağustos 2026'ya tek bir geçmişe dikiliyor ve aşağıdaki her sayı bunun üzerinden hesaplanıyor. Kolonlar: sinyal kalitesi (IC ve Newey-West t-istatistiği), taraf başına 15 bps öncesi ve sonrası long-short portföy, turnover, portföyün eşit ağırlıklı evrene göre beta'sı ve onu çıkardıktan sonra kalan alpha, ve iki anlamlılık testi: PSR (gerçek Sharpe > 0 olasılığı, tek deneme) ve deflated Sharpe oranı (aynısı, değerlendirdiğim 12 aday strateji için düzeltilmiş).

In [10]:
table, books = W.score_point_models(full)
show = ["ic_mean", "ic_tstat_nw", "sharpe_gross", "sharpe_net", "turnover", "max_dd_net",
        "beta_to_market", "alpha_bps_per_day", "alpha_tstat", "psr", "deflated_sharpe"]
market = B.run_backtest(full.point["Random"], full.targets["y_day0"], "benchmark", 0.0)["gross"]
print(f"equal-weight universe, long only, no costs: Sharpe {M.sharpe(market):.2f}, "
      f"max drawdown {M.max_drawdown(market):.0%}   (the market I hedge against)")
print(f"deflated Sharpe counts {table.attrs['n_trials']} candidate strategies as trials\n")
table[show]

equal-weight universe, long only, no costs: Sharpe -0.31, max drawdown -94%   (the market I hedge against)
deflated Sharpe counts 12 candidate strategies as trials



,ic_mean,ic_tstat_nw,sharpe_gross,sharpe_net,turnover,max_dd_net,beta_to_market,alpha_bps_per_day,alpha_tstat,psr,deflated_sharpe
model,,,,,,,,,,,
LightGBM,0.118,17.068,2.355,1.118,1.514,-0.479,-0.424,40.488,5.639,9.913e-01,2.846e-02
Transformer,0.114,16.512,2.324,1.125,1.449,-0.567,-0.431,39.312,5.520,9.919e-01,2.878e-02
GRU,0.113,16.205,2.056,0.852,1.456,-0.631,-0.449,34.330,5.063,9.657e-01,6.455e-03
Ridge,0.111,15.448,1.102,-0.058,1.490,-0.893,-0.552,17.573,2.526,4.505e-01,3.895e-06
LightGBM (smoothed),0.109,15.512,1.804,1.448,0.442,-0.575,-0.513,30.327,4.341,9.989e-01,1.146e-01
GRU (smoothed),0.108,15.351,1.783,1.455,0.411,-0.637,-0.537,29.995,4.480,9.990e-01,1.164e-01
Transformer (smoothed),0.107,15.173,1.675,1.351,0.402,-0.590,-0.518,27.818,4.062,9.980e-01,7.907e-02
Transformer-Q (q50/width),0.106,15.643,2.309,0.830,1.684,-0.547,-0.386,36.883,5.421,9.615e-01,5.839e-03
LightGBM-Q (q50/width),0.104,15.273,2.742,0.863,1.978,-0.448,-0.330,41.088,6.024,9.674e-01,6.933e-03


In [11]:
headline = ["LightGBM", "GRU", "Transformer", "Ridge", "Momentum7d", "Reversal24h", "Random"]
E.plot_equity_curves(books, headline, market);
E.plot_rolling_ic({m: M.daily_rank_ic(full.point[m], full.targets["y_day0"]) for m in headline[:5]});
E.plot_quintiles({m: B.quintile_returns(full.point[m], full.targets["y_day0"]) for m in headline});

**EN:** Reading the scorecard:
- **Beta is not alpha.** Every model's book has a beta of about -0.4 to the market: it is long BTC, ETH and BNB and short the newest, most volatile coins, which quietly bets on altcoins falling, and from 2022 to 2026 the equal-weight universe fell 94%. After regressing that out (Newey-West errors), LightGBM and the Transformer still keep ~40 bps/day with t ≈ 5.6. The edge is not just beta.
- **Trees vs. sequences.** The GRU and Transformer see the hourly path *plus* the same daily snapshot LightGBM sees, and they add nothing: the information is in the snapshot. LightGBM trains in seconds, the networks take minutes per fold on the GPU.
- **The shape of the edge.** The quintile ladders are lopsided: the bottom fifth loses ~30 bps a day, the top fifth gains ~10. Most of the money is made on the short side, in coins that are hard and expensive to short in practice.
- **Controls behave.** Random scores and a LightGBM trained on targets shuffled within each day both have IC ≈ 0, so the pipeline does not manufacture signal out of nothing.

**TR:** Skor tablosunu okurken:
- **Beta alpha değildir.** Her modelin portföyünün piyasaya betası yaklaşık -0,4: BTC, ETH ve BNB'yi long, en yeni ve en volatil coinleri short ediyor; bu, sessizce altcoinlerin düşeceğine bahse girmek demek ve 2022'den 2026'ya eşit ağırlıklı evren %94 düştü. Bunu regresyonla çıkardıktan sonra (Newey-West hatalarıyla) LightGBM ve Transformer hâlâ günde ~40 bps'yi t ≈ 5,6 ile koruyor. Avantaj yalnızca beta değil.
- **Ağaçlar ve diziler.** GRU ve Transformer saatlik yolu *artı* LightGBM'in gördüğü günlük anlık görüntünün aynısını görüyor ve hiçbir şey katmıyorlar: bilgi anlık görüntüde. LightGBM saniyeler içinde eğitiliyor, ağlar ise GPU'da fold başına dakikalar alıyor.
- **Avantajın şekli.** Quintile merdivenleri dengesiz: en alt beşte bir günde ~30 bps kaybediyor, en üst beşte bir ~10 kazanıyor. Paranın çoğu short tarafında, pratikte short etmesi zor ve pahalı coinlerde kazanılıyor.
- **Kontroller düzgün davranıyor.** Rastgele skorlar ve her gün içinde karıştırılmış hedeflerle eğitilen bir LightGBM'in ikisinin de IC'si ≈ 0; yani hat, yoktan sinyal üretmiyor.

## 8. Costs, turnover and the deflated Sharpe / Maliyetler, turnover ve deflated Sharpe

**EN:** The raw forecasts rebuild the book every day (turnover ≈ 1.5/day), which at 15 bps a side costs ~80% a year. I fixed one remedy *before looking at test results*: an exponential average of each coin's daily rank with a 3-day half-life. It keeps the ordering but moves it slowly.

**TR:** Ham tahminler portföyü her gün yeniden kuruyor (turnover ≈ günde 1,5); bu, taraf başına 15 bps ile yılda ~%80'e mal oluyor. *Test sonuçlarına bakmadan önce* tek bir çare sabitledim: her coinin günlük sırasının 3 günlük yarı ömürlü üstel ortalaması. Sıralamayı koruyor ama onu yavaş hareket ettiriyor.

In [12]:
pair = ["LightGBM", "LightGBM (smoothed)", "Transformer", "Transformer (smoothed)", "GRU", "GRU (smoothed)"]
display(table.loc[pair, ["sharpe_gross", "sharpe_net", "turnover", "max_dd_net", "psr", "deflated_sharpe"]])
curves = {m: B.cost_sensitivity(full.point[m], full.targets["y_day0"]) for m in ["LightGBM", "Transformer", "Ridge"]}
for m in ["LightGBM", "Transformer"]:
    curves[f"{m} (smoothed)"] = B.cost_sensitivity(B.smooth_signal(full.point[m]), full.targets["y_day0"])
E.plot_cost_sensitivity(curves);

,sharpe_gross,sharpe_net,turnover,max_dd_net,psr,deflated_sharpe
model,,,,,,
LightGBM,2.355,1.118,1.514,-0.479,0.991,0.028
LightGBM (smoothed),1.804,1.448,0.442,-0.575,0.999,0.115
Transformer,2.324,1.125,1.449,-0.567,0.992,0.029
Transformer (smoothed),1.675,1.351,0.402,-0.590,0.998,0.079
GRU,2.056,0.852,1.456,-0.631,0.966,0.006
GRU (smoothed),1.783,1.455,0.411,-0.637,0.999,0.116


**EN:** Smoothing cuts turnover by ~70% and lifts net Sharpe from ~1.1 to ~1.45. On its own that would look like a strategy: PSR says 99.9% that the true Sharpe is above zero. But PSR assumes this was my only idea. I evaluated 12 candidate strategies and their Sharpes varied a lot, so the best of them would look good by luck alone. The deflated Sharpe accounts for that and lands at ~0.12, far below the 0.95 I would want. **The honest conclusion is: the ranking signal is real and statistically overwhelming, but the net-of-cost edge is not proven.** Longer holding periods or execution below 15 bps a side (maker fees, VIP tiers) is what the cost curve says it would take.

**TR:** Düzleştirme turnover'ı ~%70 azaltıyor ve net Sharpe'ı ~1,1'den ~1,45'e çıkarıyor. Tek başına bu bir strateji gibi görünürdü: PSR, gerçek Sharpe'ın sıfırın üzerinde olma olasılığının %99,9 olduğunu söylüyor. Ama PSR bunun tek fikrim olduğunu varsayıyor. 12 aday strateji değerlendirdim ve Sharpe'ları çok değişkendi; dolayısıyla en iyileri yalnızca şans eseri iyi görünebilirdi. Deflated Sharpe bunu hesaba katıyor ve ~0,12'ye düşüyor; isteyeceğim 0,95'in çok altında. **Dürüst sonuç şu: sıralama sinyali gerçek ve istatistiksel olarak ezici, ama maliyet sonrası avantaj kanıtlanmış değil.** Maliyet eğrisi, bunun için daha uzun elde tutma sürelerinin ya da taraf başına 15 bps'in altında bir icranın (maker ücretleri, VIP seviyeleri) gerektiğini söylüyor.

## 9. Does it survive every regime? / Her rejimde ayakta kalıyor mu?

In [13]:
names = ["LightGBM", "Transformer", "GRU", "Ridge", "Momentum7d", "Reversal24h"]
W.per_year(full, books, names)

ic_mean                             sharpe_net                            
year              2022   2023   2024   2025   2026       2022   2023   2024   2025   2026
model                                                                                    
GRU          1.399e-01  0.106  0.110  0.106  0.098      1.095  0.309  0.827  1.055  1.161
LightGBM     1.499e-01  0.114  0.113  0.111  0.096      2.276  0.833  1.027  1.391 -0.451
Momentum7d  -2.152e-04 -0.033 -0.036 -0.022  0.031     -0.228 -0.027 -0.680 -1.568  0.966
Reversal24h  2.625e-02  0.038  0.024  0.019  0.009     -2.005 -2.660 -2.484 -3.709 -3.760
Ridge        1.403e-01  0.113  0.099  0.101  0.093      0.627  0.304 -1.065  0.149 -0.241
Transformer  1.434e-01  0.118  0.107  0.100  0.096      1.822  0.947  0.979  0.898  1.197

**EN:** The IC of every learned model (and of Ridge) is positive in every year, including 2022 (LUNA and FTX); the textbook momentum factor, by contrast, had a negative IC in 2023-2025. The net Sharpe is not: it depends on the year, which is exactly what a costly, mid-strength signal looks like.

**TR:** Öğrenilen her modelin (ve Ridge'in) IC'si her yılda pozitif; 2022 (LUNA ve FTX) dahil. Buna karşılık ders kitabı momentum faktörünün IC'si 2023-2025'te negatifti. Net Sharpe öyle değil: yıla bağlı; maliyetli, orta güçte bir sinyalin tam olarak nasıl göründüğü de bu.

## 10. Probabilistic forecasts / Olasılıksal tahminler

**EN:** For the forecasting-science side I predict nine quantiles (10% … 90%) of each coin's next-day relative return, with LightGBM (one model per quantile) and with a 9-output head on the GRU and Transformer trained on pinball loss. The baseline to beat is simple: each coin's recent volatility times the historical quantiles of volatility-standardised returns.

**TR:** Tahmin bilimi tarafı için her coinin ertesi günkü göreli getirisinin dokuz quantile'ını (%10 … %90) tahmin ediyorum: LightGBM ile (quantile başına bir model) ve pinball kaybıyla eğitilen, GRU ile Transformer üzerindeki 9 çıkışlı bir başlıkla. Geçilmesi gereken baseline basit: her coinin son dönem volatilitesi çarpı volatiliteyle standartlaştırılmış getirilerin tarihsel quantile'ları.

In [14]:
qtable, calib = W.score_quantile_models(full)
base_pinball = qtable.loc["VolScaled-Q", "pinball_mean"]
qtable["vs_baseline_%"] = 100 * (qtable["pinball_mean"] / base_pinball - 1)
display(qtable)
E.plot_calibration(calib);

,pinball_mean,coverage_80,median_width_80,vs_baseline_%
Transformer-Q,0.009,0.802,0.061,-2.083
GRU-Q,0.009,0.812,0.064,-2.074
LightGBM-Q,0.009,0.801,0.062,-1.945
VolScaled-Q,0.010,0.793,0.068,0.000


**EN:** All learned models are well calibrated (the 80% interval covers ~80% of outcomes, every quantile sits on the diagonal) and all beat the baseline, but only by ~2% in pinball loss. Most of what can be said about tomorrow's *distribution* is already in recent volatility. Using the quantiles for position sizing (median divided by interval width) did not beat the plain point forecast after costs: see the `(q50/width)` rows in the scorecard.

**TR:** Öğrenilen bütün modeller iyi kalibre (%80'lik aralık sonuçların ~%80'ini kapsıyor, her quantile köşegen üzerinde) ve hepsi baseline'ı geçiyor; ama pinball kaybında yalnızca ~%2. Yarının *dağılımı* hakkında söylenebileceklerin çoğu zaten son dönem volatilitesinde. Quantile'ları pozisyon büyüklüğü için kullanmak (medyan bölü aralık genişliği), maliyetlerden sonra düz nokta tahmini geçmedi: skor tablosundaki `(q50/width)` satırlarına bakın.

## 11. Conclusions and limitations / Sonuçlar ve sınırlamalar

**EN:**
- A daily cross-sectional ranking of liquid crypto has a strong, persistent and statistically overwhelming signal, most of it the low-volatility / lottery effect, and a tree model extracts about twice as much tradable alpha from it as a linear model.
- Hourly sequence models did not beat a tree on a daily snapshot. I would not use them here.
- Whether it makes money depends on costs. With a pre-declared turnover control it earns a net Sharpe of ~1.45 in the backtest, but after correcting for the 12 strategies I tried, that is not statistically proven.
- Limitations: the short side assumes perpetual futures and ignores funding and borrow availability for small coins; costs are a flat 15 bps with no market-impact model; fills at the hourly close are assumed; the test period is dominated by an altcoin bear market.
- What I would do next: beta- and volatility-neutral portfolio construction, funding rates as a cost and as a feature, and a longer holding period so the signal can be traded at lower turnover.

**TR:**
- Likit kriptonun günlük kesitsel sıralamasında güçlü, kalıcı ve istatistiksel olarak ezici bir sinyal var; çoğu düşük volatilite / piyango etkisi ve bir ağaç modeli, ondan doğrusal bir modele göre yaklaşık iki kat daha fazla işlem yapılabilir alpha çıkarıyor.
- Saatlik dizi modelleri, günlük anlık görüntü üzerindeki bir ağacı geçemedi. Burada onları kullanmazdım.
- Para kazandırıp kazandırmadığı maliyetlere bağlı. Önceden sabitlenmiş bir turnover kontrolüyle backtest'te ~1,45 net Sharpe kazanıyor; ama denediğim 12 strateji için düzeltmeden sonra bu, istatistiksel olarak kanıtlanmış değil.
- Sınırlamalar: short tarafı perpetual kontratları varsayıyor ve küçük coinler için funding'i ve borç bulunabilirliğini yok sayıyor; maliyetler piyasa etkisi modeli olmadan sabit 15 bps; saatlik kapanıştan işlem yapılabildiği varsayılıyor; test dönemine bir altcoin ayı piyasası hâkim.
- Sırada yapacaklarım: beta- ve volatilite-nötr portföy kurulumu, funding oranlarını hem maliyet hem özellik olarak eklemek ve sinyalin daha düşük turnover'la işlenebilmesi için daha uzun bir elde tutma süresi.